## Import and Libraries

In [1]:
from pathlib import Path
from tqdm import tqdm
import pandas as pd

import fitz                  # PDFs
import docx                  # Word
from pptx import Presentation # PowerPoint
from openpyxl import load_workbook
import json

import uuid

## Configuration 

In [2]:
DATA_PATH = Path("../data")

SUPPORTED_TYPES = {

    ".pdf",

    ".docx",

    ".pptx",

    ".xlsx",

    ".txt"

}

## Find documents

In [3]:
documents = []

for file in DATA_PATH.rglob("*"):

    if file.suffix.lower() in SUPPORTED_TYPES:

        documents.append(file)

print(f"Found {len(documents)} files.")

Found 10 files.


## Readers

In [4]:
# PDF

def read_pdf(path):
    doc = fitz.open(path)

    text = ""

    for page in doc:
        text += page.get_text()

    return text, len(doc)


# Word

def read_docx(path):

    document = docx.Document(path)

    text = "\n".join(
        p.text
        for p in document.paragraphs
    )

    return text, len(document.paragraphs)


# PowerPoint

def read_pptx(path):

    prs = Presentation(path)

    slides = []

    for slide in prs.slides:

        slide_text = []

        for shape in slide.shapes:

            if hasattr(shape, "text"):
                slide_text.append(shape.text)

        slides.append("\n".join(slide_text))

    return "\n".join(slides), len(prs.slides)


# Excel

def read_excel(path):

    workbook = load_workbook(path)

    sheets = []

    for sheet in workbook.worksheets:

        rows = []

        for row in sheet.iter_rows(values_only=True):

            rows.append(
                " ".join(
                    str(x)
                    for x in row
                    if x is not None
                )
            )

        sheets.append("\n".join(rows))

    return "\n".join(sheets), len(workbook.sheetnames)


# Text

def read_txt(path):

    return path.read_text(encoding="utf-8"), 1

## Universal Reader

In [5]:
READERS = {
    ".pdf": read_pdf,
    ".docx": read_docx,
    ".pptx": read_pptx,
    ".xlsx": read_excel,
    ".txt": read_txt
}


def ingest_document(path):

    reader = READERS[path.suffix.lower()]

    text, pages = reader(path)

    return {
        "document_id": str(uuid.uuid4()),
        "filename": path.name,
        "file_type": path.suffix.lower(),
        "pages": pages,
        "tenant": None,
        "category": None,
        "subcategory": None,
        "text": text,
        "metadata": {}
    }

## Build Knowledge Base

In [6]:
knowledge_base = []

for file in tqdm(documents):

    try:

        knowledge_base.append(

            ingest_document(file)

        )

    except Exception as e:

        print(file.name, e)

100%|██████████| 10/10 [00:00<00:00, 10.81it/s]


## Validation

In [7]:
df = pd.DataFrame(knowledge_base)

df.head()

df["metadata"] = df["metadata"].apply(json.dumps)

## Save

In [8]:
df.to_parquet(

    "../processed/knowledge_base.parquet",

    index=False

)